In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path(".")

files = [
    ("A_7_5g",  ROOT/"ASpMg_7_5g"/"ASpMg_7_5g_clean.csv"),
    ("A_15g",   ROOT/"ASpMg_15g"/"ASpMg_15g_clean.csv"),
    ("A_18g",   ROOT/"ASpMg_18g"/"ASpMg_18g_clean.csv"),
    ("A_22_5g", ROOT/"ASpMg_22_5g"/"ASpMg_22_5g_clean.csv"),
]

def load(path):
    df = pd.read_csv(path)
    # Arreglar por si quedaron nombres raros:
    df = df.rename(columns={c.lower().strip():c for c in df.columns})
    if not {"nm","A"}.issubset(df.columns):
        # Si venía como x,y entonces lo corregimos
        c0, c1 = df.columns[:2]
        df = df.rename(columns={c0:"nm", c1:"A"})
    df["nm"] = pd.to_numeric(df["nm"], errors="coerce")
    df["A"]  = pd.to_numeric(df["A"], errors="coerce")
    return df.dropna()[["nm","A"]].sort_values("nm").drop_duplicates()

# ---- MATRIZ SIN INTERPOLAR (INTERSECCIÓN EXACTA) ----

mat = load(files[0][1]).rename(columns={"A": files[0][0]})

for label, path in files[1:]:
    df = load(path).rename(columns={"A": label})
    mat = mat.merge(df, on="nm", how="inner")   # SOLO nm que existen en todos

mat = mat.sort_values("nm").reset_index(drop=True)

out = ROOT / "ASpMg_matrix.csv"
mat.to_csv(out, index=False)

print("✔ Matriz generada correctamente:")
print(out)
display(mat.head())

✔ Matriz generada correctamente:
ASpMg_matrix.csv


,nm,A_7_5g,A_15g,A_18g,A_22_5g
0,213.5,1.518,2.342,1.689,2.269
1,214.0,1.493,2.260,1.661,2.224
2,214.5,1.468,2.192,1.630,2.175
3,215.0,1.443,2.136,1.600,2.125
4,215.5,1.417,2.087,1.569,2.075
